# Metal Defect Prediction (Testing)
This notebook allows you to test the trained model on a single image.

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import os
import matplotlib.pyplot as plt

# 1. Define Model Architecture (Must match training)
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=6):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 2. Prediction Function
def predict_image(image_path, model_path='metal_defect_model_notebook.pkl', confidence_threshold=0.5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    classes = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
    
    # Load model
    model = SimpleCNN(num_classes=len(classes))
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    # Preprocess image
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]
        
    max_prob, predicted_idx = torch.max(probabilities, 0)
    
    if max_prob.item() < confidence_threshold:
        result = "Good (المعدن ده كويس)"
    else:
        result = classes[predicted_idx.item()]
        
    return result, max_prob.item(), image

## Run Prediction
Replace the value of `img_path` below with the path to your image file.

In [ ]:
# EDIT THIS PATH:
img_path = r'E:\Semester 8\Advanced Methods\Project\data\NEU-DET\validation\images\crazing_241.jpg'

if os.path.exists(img_path):
    label, confidence, img = predict_image(img_path)
    
    # Show result
    plt.imshow(img)
    plt.title(f"Prediction: {label} ({confidence*100:.2f}%)")
    plt.axis('off')
    plt.show()
    
    print(f"Result: {label}")
    print(f"Confidence: {confidence:.4f}")
else:
    print(f"File not found at: {img_path}")